# 07 · JDM compared with CANDLE, SAVI, AGS and UAID: each disease against its own lab's healthy controls

**Run order:** 02 → 03 → 04 → 05 → 06 → **07 (this notebook)**. (04b, the all-sources correction, is separate; this notebook does not need it.) Kernel: **R (WGCNA)**. Needs notebook 05 (differential expression) and 04 (corrected counts) to have run.

**The question.** How does the blood signature of JDM compare with the interferonopathies (CANDLE, SAVI, AGS, UAID)?

**Why it is done this way.** The JDM samples come from a different lab and a different pipeline (RPKM) than your NIH samples (raw counts). If you simply pool them, the biggest difference you see is the lab. What *can* be compared fairly is **each disease against the healthy controls of its own lab**:
- JDM against the JDM lab's healthy controls (from the key sheet, `Diagnosis = HC`);
- CANDLE, SAVI, AGS, UAID against the NIH healthy controls (notebook 05).

Both give a list of genes up or down in disease *relative to healthy, in the same lab*. Those lists can be put side by side without the lab difference in the way. This notebook does three things with them:
1. **Gene by gene:** log fold change in JDM against log fold change in each NIH disease (rank correlation, how many top genes are shared, how many go the same direction).
2. **Interferon score:** each sample's score, measured against its own lab's healthy controls, for every group side by side.
3. **Clusters:** every disease sample (JDM and NIH) described by how far each gene sits from its own lab's healthy controls, then clustered together.
4. *(Moved.)* The one pooled view with the lab difference removed is now notebook **04b**, which also checks that the correction worked.

**What is *not* done:** statistics are never run on the pooled values, so "JDM vs CANDLE directly" is not tested. This notebook never puts the two labs on one scale; notebook 04b does, for plots only. Because the two datasets use different scales (voom log CPM against quantile-normalised log RPKM), fold changes are compared by **rank and direction**, not by size.

**JDM samples dropped:** any JDM-lab sample that is the *same library* as an NIH sample (its ID ends an NIH sample name), and, if `DROP_SAME_PERSON`, any JDM sample of a person already in the NIH set. Counts are printed.

**Reads:** `/mnt/file-systems/4b_rpkm.csv`, the optional key sheet; notebook 05's `de_*_vs_HC.csv`; notebook 04's corrected counts; notebook 03's sample table.
**Writes (all CSV):** `rnaseq_out_v2/jdm_comparison/`: `de_JDM_vs_HC_jdm_lab.csv`, `comparison_summary.csv`, `compare_JDM_vs_<disease>.csv`, `ifn_score_by_group.csv`, `sample_clusters_with_jdm.csv`.

In [ ]:
RESULTS <- "/home/jovyan/session_data/Attempt0/endotype_results"
IN_DIR  <- file.path(RESULTS, "rnaseq_out_v2")
DE_DIR  <- file.path(IN_DIR, "differential_expression")
BC_DIR  <- file.path(IN_DIR, "batch_correction")
OUT_DIR <- file.path(IN_DIR, "jdm_comparison")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

MNT      <- "/mnt/file-systems/"
JDM_RPKM <- paste0(MNT, "4b_rpkm.csv")                   # rows Cohort, ID, then genes by symbol (RPKM)
QUANTILE_NORMALISE <- TRUE       # after log2(RPKM + 1): quantile-normalise across the JDM samples, the lab's own recipe (their 4a file)
JDM_KEY  <- ""                   # OPTIONAL key sheet (Initials = patient block; NIH Enroll ID = same-person check). "" = not used
HC_PATTERN  <- "^(hc|relative-hc|healthy|control)"   # a label (Cohort row of 4b_rpkm.csv) matching this = healthy control
JDM_PATTERN <- "jdm|dermatomyositis"                 # a label matching this = JDM
PATIENT_COL <- "Initials"        # key-sheet column that identifies a JDM patient (used as the patient block); blank = sample is its own patient
DROP_SAME_PERSON <- TRUE         # drop JDM samples of a person already in the NIH set
ALPHA   <- 0.05                  # adjusted p-value cut-off for "top genes"
LFC     <- 1                     # |log2 FC| cut-off, NIH side (as in notebook 05)
JDM_LFC <- 0.5                   # |log2 FC| cut-off, JDM side: quantile normalisation squeezes fold changes, so a lower cut-off
K_CLUSTERS <- 4                  # sample groups in the final clustering
RECODE <- c("Undifferentiated Interferonopathy" = "UAID")
GENE_SET_DIR <- "/home/jovyan/session_data/endotype-transcripts-007/genes"   # ifn-response-genes-28.txt (one symbol per line)

for (f in c(JDM_RPKM, if (nzchar(JDM_KEY)) JDM_KEY, file.path(BC_DIR, "counts_combatseq_eligible.csv"), file.path(DE_DIR, "summary_comparisons.csv")))
  if (!file.exists(f)) stop("Missing file: ", f)
suppressMessages({library(limma); library(edgeR)})

norm_id <- function(x) { x <- toupper(trimws(as.character(x))); x[is.na(x)] <- ""; x <- sub("\\.0$", "", x)
  x <- gsub("[-_ ]", "", x); x <- sub("^CHLD", "LD", x); x[x %in% c("NAN", "NONE", "NA")] <- ""; x }
by_symbol <- function(M) { o <- order(rowMeans(M, na.rm = TRUE), decreasing = TRUE); M <- M[o, , drop = FALSE]
  M[!is.na(rownames(M)) & rownames(M) != "" & !duplicated(rownames(M)), , drop = FALSE] }   # one row per symbol: the best-expressed

## The NIH samples (notebooks 03 and 04)
Corrected counts are turned into log2 counts per million (display values, as in notebook 06) and reduced to one row per gene symbol. `diagnosis` is the NIH diagnosis, with `HC` for the NIH healthy controls.

In [ ]:
cd <- read.csv(file.path(BC_DIR, "counts_combatseq_eligible.csv"), check.names = FALSE)
M <- as.matrix(cd[, !(colnames(cd) %in% c("Gene_ID", "Gene_Name"))]); storage.mode(M) <- "numeric"; rownames(M) <- cd$Gene_ID
s <- read.csv(file.path(IN_DIR, "rnaseq_samples_eligible.csv"), check.names = FALSE, colClasses = "character")
s <- s[match(colnames(M), s$sample_id), ]
dx <- s$dx_clean; hit <- dx %in% names(RECODE); dx[hit] <- unname(RECODE[dx[hit]]); s$diagnosis <- dx
nih <- cpm(calcNormFactors(DGEList(M)), log = TRUE); rownames(nih) <- cd$Gene_Name
nih <- by_symbol(nih)
print(table(NIH_diagnosis = s$diagnosis))
c(NIH_samples = ncol(nih), genes_by_symbol = nrow(nih))

## The JDM-lab samples
`4b_rpkm.csv` holds RPKM by gene symbol (rows `Cohort` and `ID`, then the genes). It is turned into log2(RPKM + 1) and quantile-normalised across the JDM samples (`QUANTILE_NORMALISE`), which is how the lab made its own log2 file. The file's own `Cohort` row says which sample is JDM and which is a healthy control (the key sheet is optional) (R added `...2`-style suffixes to the labels, which are removed). Then the overlaps with the NIH set are dropped, as described at the top.

In [ ]:
raw <- read.csv(JDM_RPKM, check.names = FALSE, colClasses = "character")
gcol <- colnames(raw)[1]
is_meta <- raw[[gcol]] %in% c("Cohort", "ID")
ids <- unlist(raw[raw[[gcol]] == "ID", -1], use.names = FALSE)
cohort <- if ("Cohort" %in% raw[[gcol]]) unlist(raw[raw[[gcol]] == "Cohort", -1], use.names = FALSE) else rep("", length(ids))
cohort[is.na(cohort)] <- ""
E <- as.matrix(raw[!is_meta, -1]); storage.mode(E) <- "numeric"
rownames(E) <- raw[[gcol]][!is_meta]; colnames(E) <- ids
E <- log2(E + 1)
if (QUANTILE_NORMALISE) { full <- rowSums(is.na(E)) == 0; E[full, ] <- normalizeQuantiles(E[full, , drop = FALSE]) }
E <- by_symbol(E)

clean_label <- function(x) trimws(sub("\\.\\.\\.[0-9]+$", "", x))
classify <- function(x) ifelse(grepl(HC_PATTERN, x, ignore.case = TRUE), "HC",
                        ifelse(grepl(JDM_PATTERN, x, ignore.case = TRUE), "JDM", NA))
group <- classify(clean_label(cohort))
if (!any(group == "HC", na.rm = TRUE)) group <- classify(clean_label(ids))
cat("How the labels in 4b_rpkm.csv (Cohort row) were read:\n")
print(table(label_in_file = ifelse(cohort == "", "(blank)", clean_label(cohort)), classified_as = ifelse(is.na(group), "NEITHER", group)))
jm <- data.frame(sample = colnames(E), group = group, Initials = "", `NIH Enroll ID` = "", check.names = FALSE, stringsAsFactors = FALSE)
if (nzchar(JDM_KEY)) {                     # optional key sheet: patient initials and NIH enrollment ID
  key <- read.csv(JDM_KEY, check.names = FALSE, colClasses = "character")
  colnames(key) <- trimws(colnames(key)); key <- key[!duplicated(key$ID), ]
  kk <- key[match(jm$sample, key$ID), , drop = FALSE]
  if ("Initials" %in% colnames(kk)) jm$Initials <- ifelse(is.na(kk$Initials), "", kk$Initials)
  if ("NIH Enroll ID" %in% colnames(kk)) jm[["NIH Enroll ID"]] <- ifelse(is.na(kk[["NIH Enroll ID"]]), "", kk[["NIH Enroll ID"]])
}

same_lib <- vapply(jm$sample, function(i) any(endsWith(s$sample_id, paste0("-", i))), logical(1))
pid_nih  <- norm_id(jm[["NIH Enroll ID"]])
same_person <- DROP_SAME_PERSON & pid_nih != "" & pid_nih %in% s$patient_key
keep <- !is.na(jm$group) & !same_lib & !same_person
cat("JDM-lab samples:", ncol(E), "| neither JDM nor HC:", sum(is.na(jm$group)), "\n")
cat("dropped: same library as an NIH sample:", sum(same_lib), "| same person as an NIH patient:", sum(same_person & !same_lib), "\n")
E <- E[, keep, drop = FALSE]; jm <- jm[keep, , drop = FALSE]
print(table(JDM_lab_group = jm$group))
if (sum(jm$group == "HC") < 3 || sum(jm$group == "JDM") < 3) stop("Need at least 3 JDM and 3 healthy-control samples in the JDM lab after the drops above.")
c(JDM_samples = sum(jm$group == "JDM"), JDM_lab_HC = sum(jm$group == "HC"), genes_by_symbol = nrow(E))

## JDM against its own healthy controls
limma on the log2 values (genes with a value in every sample), design `~ 0 + group`. If a patient has more than one sample (by `PATIENT_COL`), the patient is a block, as in notebook 05. Positive log fold change = higher in JDM.

In [ ]:
pid <- norm_id(jm[[PATIENT_COL]]); pid[pid == ""] <- paste0("sample_", jm$sample[pid == ""])
grp <- factor(jm$group, levels = c("HC", "JDM"))
design <- model.matrix(~ 0 + grp); colnames(design) <- c("HC", "JDM")
Ek <- E[rowSums(is.na(E)) == 0, , drop = FALSE]
if (any(table(pid) > 1)) {
  cor1 <- duplicateCorrelation(Ek, design, block = pid)$consensus
  fit  <- lmFit(Ek, design, block = pid, correlation = cor1)
  cat("patient correlation:", round(cor1, 3), "\n")
} else fit <- lmFit(Ek, design)
fit2 <- eBayes(contrasts.fit(fit, makeContrasts(JDM - HC, levels = design)), trend = TRUE, robust = TRUE)
jdm <- topTable(fit2, coef = 1, number = Inf, sort.by = "P"); jdm$gene_name <- rownames(jdm)
write.csv(jdm, file.path(OUT_DIR, "de_JDM_vs_HC_jdm_lab.csv"), row.names = FALSE)
c(genes_tested = nrow(jdm), adj_p_below_cutoff = sum(jdm$adj.P.Val < ALPHA),
  top_up = sum(jdm$adj.P.Val < ALPHA & jdm$logFC >= JDM_LFC), top_down = sum(jdm$adj.P.Val < ALPHA & jdm$logFC <= -JDM_LFC))

## 1. Gene by gene: JDM against each NIH disease
Each NIH disease's `de_<disease>_vs_HC.csv` from notebook 05 (NIH disease against NIH healthy controls) is matched to JDM against JDM healthy controls **by gene symbol**. For each disease: how many genes are compared, the **rank correlation** of the two fold-change lists, how many top genes each side has, how many are top in both, and how many of those go the **same direction**. The scatter plots put the NIH log fold change on the x axis and the JDM one on the y axis; interferon genes (if the list is found) are in red.

In [ ]:
read_set <- function(f) { p <- file.path(GENE_SET_DIR, f); if (!file.exists(p)) return(NULL)
  x <- trimws(readLines(p)); x[x != "" & !startsWith(x, "#")] }
irg <- read_set("ifn-response-genes-28.txt")

files <- list.files(DE_DIR, pattern = "^de_.*_vs_HC\\.csv$", full.names = TRUE)
if (!length(files)) stop("No de_<disease>_vs_HC.csv in ", DE_DIR, ": notebook 05 needs HC among its groups.")
rows <- list(); keepm <- list()
for (f in files) {
  dis <- sub("_vs_HC$", "", sub("^de_", "", sub("\\.csv$", "", basename(f))))
  n <- read.csv(f); n <- n[order(-n$AveExpr), ]; n <- n[!duplicated(n$gene_name), ]
  m <- merge(n[, c("gene_name", "logFC", "adj.P.Val")], jdm[, c("gene_name", "logFC", "adj.P.Val")], by = "gene_name", suffixes = c("_nih", "_jdm"))
  sn <- m$adj.P.Val_nih < ALPHA & abs(m$logFC_nih) >= LFC
  sj <- m$adj.P.Val_jdm < ALPHA & abs(m$logFC_jdm) >= JDM_LFC
  both <- sn & sj; same <- sign(m$logFC_nih) == sign(m$logFC_jdm)
  rows[[dis]] <- data.frame(disease = dis, genes_compared = nrow(m),
    rank_correlation_logFC = round(cor(m$logFC_nih, m$logFC_jdm, method = "spearman"), 3),
    top_genes_nih = sum(sn), top_genes_jdm = sum(sj), top_in_both = sum(both),
    top_in_both_same_direction = sum(both & same),
    pct_same_direction_in_either = if (any(sn | sj)) round(100 * mean(same[sn | sj])) else NA)
  m$ifn_gene <- if (is.null(irg)) FALSE else m$gene_name %in% irg
  keepm[[dis]] <- m
  write.csv(m, file.path(OUT_DIR, sprintf("compare_JDM_vs_%s.csv", dis)), row.names = FALSE)
}
cmp <- do.call(rbind, rows); rownames(cmp) <- NULL
write.csv(cmp, file.path(OUT_DIR, "comparison_summary.csv"), row.names = FALSE)
cmp
options(repr.plot.width = 5 * length(keepm), repr.plot.height = 5)
par(mfrow = c(1, length(keepm)), mar = c(4, 4, 3, 1))
for (dis in names(keepm)) { m <- keepm[[dis]]
  plot(m$logFC_nih, m$logFC_jdm, pch = 16, cex = 0.35, col = adjustcolor("grey30", 0.4),
       xlab = sprintf("%s vs NIH healthy (log2 FC)", dis), ylab = "JDM vs JDM-lab healthy (log2 FC)",
       main = sprintf("%s: rho = %.2f", dis, cmp$rank_correlation_logFC[cmp$disease == dis]), cex.main = 0.9)
  abline(h = 0, v = 0, lty = 2, col = "grey60")
  if (any(m$ifn_gene)) points(m$logFC_nih[m$ifn_gene], m$logFC_jdm[m$ifn_gene], pch = 16, cex = 0.9, col = "#B2182B")
}

## 2. Interferon score, measured against each lab's own healthy controls
The score of a sample is the mean over the 28 interferon response genes of its z-score, where each gene's z-score is taken **against the healthy controls of the same lab** (value minus the lab's healthy mean, divided by the lab's healthy standard deviation). A score near 0 is healthy-like; higher means more interferon signal. This is the fair way to put JDM and the NIH diseases on one axis. If the gene list is not found, this part is skipped.

In [ ]:
if (is.null(irg)) {
  cat("Gene list not found in", GENE_SET_DIR, "- set GENE_SET_DIR to the repo's genes/ folder.\n"); ifn <- NULL
} else {
  g <- Reduce(intersect, list(irg, rownames(nih), rownames(E)))
  cat("interferon genes in both datasets:", length(g), "of", length(irg), "\n")
  sc <- function(X, hc_cols) { H <- X[g, hc_cols, drop = FALSE]
    Z <- (X[g, , drop = FALSE] - rowMeans(H)) / pmax(apply(H, 1, sd), 0.1); colMeans(Z, na.rm = TRUE) }
  sn <- sc(nih, which(s$diagnosis == "HC")); sj <- sc(E, which(jm$group == "HC"))
  ifn <- data.frame(group = c(ifelse(s$diagnosis == "HC", "HC (NIH)", s$diagnosis), ifelse(jm$group == "HC", "HC (JDM lab)", "JDM")),
                    lab = rep(c("NIH", "JDM lab"), c(length(sn), length(sj))), ifn_score = c(sn, sj))
  tab <- aggregate(ifn_score ~ group, ifn, function(x) c(n = length(x), median = round(median(x), 2)))
  tab <- data.frame(group = tab$group, samples = tab$ifn_score[, "n"], median_ifn_score = tab$ifn_score[, "median"])
  print(tab[order(tab$median_ifn_score), ], row.names = FALSE)
  write.csv(tab, file.path(OUT_DIR, "ifn_score_by_group.csv"), row.names = FALSE)
  options(repr.plot.width = 10, repr.plot.height = 5)
  ord <- tab$group[order(tab$median_ifn_score)]
  par(mar = c(7, 4, 2, 1)); boxplot(ifn_score ~ factor(group, levels = ord), ifn, las = 2, xlab = "", ylab = "interferon score (z against own lab's healthy controls)",
                                    col = ifelse(grepl("JDM", ord), "#E6AB02", ifelse(grepl("HC", ord), "grey80", "#7570B3")))
  abline(h = 0, lty = 2, col = "grey50")
}

## 3. Clusters of all disease samples, JDM included
Genes: the top genes of JDM against healthy (the 25 with the smallest adjusted p) plus the top 25 of each NIH disease against healthy, **kept only if present in both datasets**. Each sample is described by how far each gene sits from the healthy controls of its **own** lab, in units of that lab's healthy standard deviation, then clipped to ±4. Samples (healthy controls included, for reference) and genes are clustered with Ward's method. **Read the tables:** a cluster that mixes JDM and NIH diseases shares a signature; a cluster that is purely one lab may still reflect the lab.

In [ ]:
top_nih <- unique(unlist(lapply(files, function(f) { n <- read.csv(f); head(n$gene_name[n$adj.P.Val < ALPHA & abs(n$logFC) >= LFC], 25) })))
top_jdm <- head(jdm$gene_name[jdm$adj.P.Val < ALPHA & abs(jdm$logFC) >= JDM_LFC], 25)
gg <- intersect(union(top_nih, top_jdm), intersect(rownames(nih), rownames(E)))
cat("genes used:", length(gg), "(NIH top:", length(top_nih), "| JDM top:", length(top_jdm), ")\n")
if (length(gg) < 5) stop("Too few shared top genes to cluster; check notebook 05 and the JDM comparison above.")

rel <- function(X, hc_cols) { H <- X[gg, hc_cols, drop = FALSE]
  pmin(pmax((X[gg, , drop = FALSE] - rowMeans(H)) / pmax(apply(H, 1, sd), 0.1), -4), 4) }
R <- cbind(rel(nih, which(s$diagnosis == "HC")), rel(E, which(jm$group == "HC")))
meta <- data.frame(group = c(ifelse(s$diagnosis == "HC", "HC (NIH)", s$diagnosis), ifelse(jm$group == "HC", "HC (JDM lab)", "JDM")),
                   lab = rep(c("NIH", "JDM lab"), c(ncol(nih), ncol(E))), stringsAsFactors = FALSE)
H <- t(R); ok <- colSums(is.na(H)) == 0; H <- H[, ok, drop = FALSE]
hr <- hclust(dist(H), method = "ward.D2"); hc <- hclust(dist(t(H)), method = "ward.D2")
cl <- cutree(hr, K_CLUSTERS)
cat("\ncluster x group\n");  print(table(cluster = cl, group = meta$group))
cat("\ncluster x lab\n");    print(table(cluster = cl, lab = meta$lab))

g_lev <- sort(unique(meta$group)); g_col <- setNames(hcl.colors(length(g_lev), "Dark 3"), g_lev)
l_col <- c(NIH = "#1B9E77", "JDM lab" = "#E6AB02")
options(repr.plot.width = 12, repr.plot.height = 1.4); par(mar = c(0, 0, 0, 0)); plot.new()
legend("topleft", title = "group", legend = names(g_col), fill = g_col, bty = "n", ncol = 3, cex = 0.9)
legend("topright", title = "lab", legend = names(l_col), fill = l_col, bty = "n", cex = 0.9)
options(repr.plot.width = 12, repr.plot.height = 9)
heatmap(H, Rowv = as.dendrogram(hr), Colv = as.dendrogram(hc), scale = "none",
        RowSideColors = unname(g_col[meta$group]),
        labRow = rep("", nrow(H)), labCol = colnames(H), cexCol = 0.6, margins = c(8, 2), col = hcl.colors(51, "Blue-Red 3"),
        main = sprintf("Distance from each lab's healthy controls: %d genes x %d samples; ward.D2", ncol(H), nrow(H)), cex.main = 0.8)
write.csv(data.frame(sample_id = c(s$sample_id, jm$sample), group = meta$group, lab = meta$lab, cluster = cl),
          file.path(OUT_DIR, "sample_clusters_with_jdm.csv"), row.names = FALSE)
list.files(OUT_DIR)

## 4. The pooled view now lives in notebook 04b
It used to be here. It moved to **04b_NIH_JDM_batch_correction_all_sources**, which does the same ComBat step for all seven sources (the facilities and the JDM lab), says which sources can and cannot be corrected, and checks that the interferon signal survived. This notebook stays what it was meant to be: each disease against its own lab's healthy controls.

---
## What to check
1. **Dropped samples:** how many JDM samples were the same library or person as NIH samples. Those are removed so one person is not counted twice.
2. **Rank correlation and shared top genes** (table in section 1): a clearly positive correlation with many shared top genes in the same direction means JDM and that disease share a signature relative to healthy. Near 0 means they do not.
3. **Interferon score:** healthy controls of both labs should sit near 0. JDM and the interferonopathies should sit higher. If the two healthy groups themselves are far apart, the lab difference is large and the comparisons should be read with more care.
4. **Clusters:** JDM samples mixing with CANDLE/SAVI/UAID samples (cluster × group) is the signal of interest; clusters split purely by lab are not.

**Limits.** The lab difference is removed only by measuring each disease against its own lab's healthy controls. It is assumed that the healthy controls of each lab are comparable healthy blood; with only a few controls per lab, the z-scores are noisy. The JDM files are assumed (by you) to be whole blood; nothing in the files shows it.

## References
1. Ritchie ME et al. limma powers differential expression analyses for RNA-sequencing and microarray studies. *Nucleic Acids Res* 2015;43:e47.
2. Smyth GK, Michaud J, Scott HS. *Bioinformatics* 2005;21:2067-2075 (patient block).
3. Law CW et al. voom. *Genome Biol* 2014;15:R29.
4. Ward JH. *J Am Stat Assoc* 1963;58:236-244.
5. Kim H et al. *J Interferon Cytokine Res* 2018;38:171-185; de Jesus AA et al. *J Clin Invest* 2020;130:1669-1682 (interferon response genes).